# Show Evaluation 2 Results
Shows the present results of eval 2.

## 1. Setup

In [1]:
import os
from pathlib import Path
import json
import pandas as pd
from src.food_rag.evaluation import retriever_evaluator
from src.food_rag.evaluation.retriever_evaluator import RUN_PATH
from src.food_rag.utils import hummus_docify
from src.food_rag.utils import llm_helper
from src.food_rag.utils import qa_helper
from src.persona import qa_generator
from src.food_rag.evaluation import eval_parser
from src.utils import load_util
from llama_index.core.evaluation import CorrectnessEvaluator
from llama_index.core.evaluation import FaithfulnessEvaluator
from llama_index.core.evaluation import GuidelineEvaluator
from llama_index.core.evaluation import RelevancyEvaluator

# Set working directory
os.chdir("....")  # Change to the root directory of the project
cwd = os.getcwd()
print("Current Working Directory:", cwd)

Current Working Directory: D:\DataspellProjects\foodRAG


In [2]:
## Output Paths
data_path = './data'
qa_name = 'big_1'
run_name = 'eval_2_4'
load_util.ensure_dir(data_path + "/runs/" + run_name)

In [3]:
# Set eval params
qa_set_size = 1000
correctness_score_threshold = 3.0  # The threshold for the correctness evaluator between 0 and 5 where 5 is the best score
guidelines = "The response must recommend a meal plan. A meal plan consists of one recommended recipe for each meal type (e.g. breakfast, lunch, dinner). The meal plan must avoid disliked things or allergies, and can optionally include liked things. It should be understandable why the meal plan was recommended."  # The guidelines applied to the guideline evaluator

In [4]:
# Load the current recipes data
hummus_data = load_util.get_current_hummus(data_path + "/full_hummus")

In [5]:
# # Load question-answer pairs & their embeddings (only the recipe qa's)
# question_answer_sets = qa_generator.load_qa_set(data_path, sub_dir_name=qa_name, recipes_df=hummus_data['recipes_df'])['recipe_question_answer_set']
#
# # Clean the qa set and add the embeddings
# qa_set = qa_helper.get_best_questions(question_answer_sets, limit_size=qa_set_size, discard_empty_qas=True)
# qa_set = qa_helper.get_recipes_from_ids(qa_set, hummus_data['recipes_df'])
# qa_set = qa_helper.get_contexts_qa_set(qa_set)
#
# del question_answer_sets  # delete the question answer sets to free up memory

## 2. Evaluation
### 2.1 Simple recommender
#### 2.1.0 Recipe Queries + BM25 Recommender

In [6]:
# Load the response
responses_0 = load_util.load_file(data_path + "/runs/" + run_name + "/responses_0.pkl")

# Load the results
serialized_results_0 = load_util.load_file(data_path + "/runs/" + run_name + "/bm25_eval_results.pkl")

# Print the results
llm_helper.print_serialized_scores(serialized_results_0)

Metric: faithfulness
Overall Score: 0.584 (584.0/1000)
Number of invalid Results: 0


Metric: relevancy
Overall Score: 0.577 (577.0/1000)
Number of invalid Results: 0


Metric: correctness
Overall Score: 1.9919191919191919 (1972.0/990)
Number of invalid Results: 10




#### 2.1.1 Recipe Queries + tfidf Recommender

In [6]:
# # Load the response
# responses_1 = load_util.load_file(data_path + "/runs/" + run_name + "/responses_1.pkl")
#
# # Load the results
# serialized_results_1 = load_util.load_file(data_path + "/runs/" + run_name + "/tfidf_eval_results.pkl")
#
# # Print the results
# llm_helper.print_serialized_scores(serialized_results_1)

#### 2.1.2 Recipe Queries + keyword overlap Recommender

In [7]:
# # Load the response
# responses_2 = load_util.load_file(data_path + "/runs/" + run_name + "/responses_2.pkl")
#
# # Load the results
# serialized_results_2 = load_util.load_file(data_path + "/runs/" + run_name + "/keyword_overlap_eval_results.pkl")
#
# # Print the results
# llm_helper.print_serialized_scores(serialized_results_2)

#### 2.1.3 Recipe Queries + jaccard Recommender


In [8]:
# Load the response
# responses_3 = load_util.load_file(data_path + "/runs/" + run_name + "/responses_3.pkl")
#
# # Load the results
# serialized_results_3 = load_util.load_file(data_path + "/runs/" + run_name + "/jaccard_eval_results.pkl")
#
# # Print the results
# llm_helper.print_serialized_scores(serialized_results_3)

#### 2.1.4 Recipe Queries + bigram overlap Recommender


In [9]:
# # Load the response
# responses_4 = load_util.load_file(data_path + "/runs/" + run_name + "/responses_4.pkl")
#
# # Load the results
# serialized_results_4 = load_util.load_file(data_path + "/runs/" + run_name + "/bigram_overlap_eval_results.pkl")
#
# # Print the results
# llm_helper.print_serialized_scores(serialized_results_4)

#### 2.1.5 Recipe Queries + bow cosine Recommender


In [10]:
# # Load the response
# responses_5 = load_util.load_file(data_path + "/runs/" + run_name + "/responses_5.pkl")
#
# # Load the results
# serialized_results_5 = load_util.load_file(data_path + "/runs/" + run_name + "/bow_cosine_eval_results.pkl")
#
# # Print the results
# llm_helper.print_serialized_scores(serialized_results_5)

#### 2.1.6 Recipe Queries + doc2vec Recommender


In [11]:
# # Load the response
# responses_6 = load_util.load_file(data_path + "/runs/" + run_name + "/responses_6.pkl")
#
# # Load the results
# serialized_results_6 = load_util.load_file(data_path + "/runs/" + run_name + "/doc2vec_eval_results.pkl")
#
# # Print the results
# llm_helper.print_serialized_scores(serialized_results_6)

#### 2.1.7 Recipe Queries + embedding similarity Recommender


In [12]:
# # Load the response
# responses_7 = load_util.load_file(data_path + "/runs/" + run_name + "/responses_7.pkl")
#
# # Load the results
# serialized_results_7 = load_util.load_file(data_path + "/runs/" + run_name + "/embedding_similarity_eval_results.pkl")
#
# # Print the results
# llm_helper.print_serialized_scores(serialized_results_7)

### 2.2 Recipe Recommendation RAG
### 2.2.1 Recipe Recommendation RAG

In [13]:
# # Load the response
# rag_responses = load_util.load_file(data_path + "/runs/" + run_name + "/rag_responses.pkl")
#
# # Load the results
# serialized_results_8 = load_util.load_file(data_path + "/runs/" + run_name + "/rag_eval_results.pkl")
#
# # Print the results
# llm_helper.print_serialized_scores(serialized_results_8)

Metric: faithfulness
Overall Score: 0.932 (932.0/1000)
Number of invalid Results: 0


Metric: relevancy
Overall Score: 0.777 (777.0/1000)
Number of invalid Results: 0


Metric: guideline
Overall Score: 0.0 (0.0/1000)
Number of invalid Results: 0


Metric: correctness
Overall Score: 2.8755040322580645 (2852.5/992)
Number of invalid Results: 8




### 2.2.2 Recipe Recommendation RAG with Pandas-To-Text

In [7]:
# Load the response
rag_responses_2 = load_util.load_file(data_path + "/runs/" + run_name + "/rag_responses_2.pkl")

# Load the results
serialized_results_11 = load_util.load_file(data_path + "/runs/" + run_name + "/rag_eval_results_2.pkl")

# Print the results
llm_helper.print_serialized_scores(serialized_results_11)

Metric: faithfulness
Overall Score: 0.909 (909.0/1000)
Number of invalid Results: 0


Metric: relevancy
Overall Score: 0.783 (783.0/1000)
Number of invalid Results: 0


Metric: guideline
Overall Score: 0.0 (0.0/1000)
Number of invalid Results: 0


Metric: correctness
Overall Score: 3.0014000000000007 (3001.4000000000005/1000)
Number of invalid Results: 0




### 2.3 Heuristic Agents
#### 2.3.1 Heuristic Agent #1

In [14]:
# # Load the response
# heuristic_responses_1 = load_util.load_file(data_path + "/runs/" + run_name + "/heuristic_responses_1.pkl")
#
# # Load the results
# serialized_results_9 = load_util.load_file(data_path + "/runs/" + run_name + "/heuristic_agent_eval_results_1.pkl")
#
# # Print the results
# llm_helper.print_serialized_scores(serialized_results_9)

Metric: faithfulness
Overall Score: 0.86 (860.0/1000)
Number of invalid Results: 0


Metric: relevancy
Overall Score: 0.752 (752.0/1000)
Number of invalid Results: 0


Metric: guideline
Overall Score: 0.022 (22.0/1000)
Number of invalid Results: 0


Metric: correctness
Overall Score: 3.2993920972644375 (3256.5/987)
Number of invalid Results: 13




#### 2.3.1 Heuristic Agent #2 (short version)

In [15]:
# # Load the response
# heuristic_responses_2 = load_util.load_file(data_path + "/runs/" + run_name + "/heuristic_responses_2.pkl")
#
# # Load the results
# serialized_results_10 = load_util.load_file(data_path + "/runs/" + run_name + "/heuristic_agent_eval_results_2.pkl")
#
# # Print the results
# llm_helper.print_serialized_scores(serialized_results_10)

Metric: faithfulness
Overall Score: 0.865 (865.0/1000)
Number of invalid Results: 0


Metric: relevancy
Overall Score: 0.873 (873.0/1000)
Number of invalid Results: 0


Metric: guideline
Overall Score: 0.002 (2.0/1000)
Number of invalid Results: 0


Metric: correctness
Overall Score: 2.930340557275542 (2839.5/969)
Number of invalid Results: 31




# 3. Summary

In [10]:
# Summarize the evaluation results in a df
results_df = retriever_evaluator.summarize_analyse_results(
    data={
        '0_bm25': {'serialized_results': serialized_results_0, 'responses': responses_0},
        # '1_tfidf': {'serialized_results': serialized_results_1, 'responses': responses_1},
        #'2_Keyword_overlap': {'serialized_results': serialized_results_2, 'responses': responses_2},
        #'3_Jaccard': {'serialized_results': serialized_results_3, 'responses': responses_3},
        #'4_bigram_overlap': {'serialized_results': serialized_results_4, 'responses': responses_4},
        # '5_bow_cosine': {'serialized_results': serialized_results_5, 'responses': responses_5},
        #'6_doc2vec': {'serialized_results': serialized_results_6, 'responses': responses_6},
        #'7_embedding_similarity': {'serialized_results': serialized_results_7, 'responses': responses_7},
        #'8_RAG': {'serialized_results': serialized_results_8, 'responses': rag_responses},
        #'9_Heuristic#1': {'serialized_results': serialized_results_9, 'responses': heuristic_responses_1},
        #'10_Heuristic#2': {'serialized_results': serialized_results_10, 'responses': heuristic_responses_2},
        '11_Pandas-To-Text': {'serialized_results': serialized_results_11, 'responses': rag_responses_2},
    },
    key="overall_score", # pull just this field from each metric block
    k=5,
    data_path=data_path, run_name=run_name,
    normalize_scores={"correctness": ((1, 5),  True)}  # correctness is between 1 and 5, so we normalize it to be between 0 and 1
)
results_df

,faithfulness,relevancy,correctness,guideline,correctness_norm,meta_llm_parsing_usage_counter,meta_llm_usage_counter,meta_llm_sequential_usage_counter
0_bm25,0.584,0.577,1.991919,NaN,0.24798,NaN,NaN,NaN
11_Pandas-To-Text,0.909,0.783,3.001400,0.0,0.50035,0.0,1.0,1.0


In [11]:
# Store the evaluation results
load_util.store_file(data=results_df, path=Path(data_path, RUN_PATH, run_name, 'evaluation_results'))

# 4. More Analysis

In [18]:
# Max scores
max_scores = {
    metric:  # e.g. "faithfulness"
        max(  # take the biggest score
            (r["score"]  # every score in this block
             for r in block["results"]
             if r.get("score") is not None),  # keep only real numbers
            default=None  # in case all scores are None
        ) for metric, block in serialized_results_10.items()
}

print(json.dumps(max_scores, indent=2, sort_keys=True))

# Note: correctness returns a value between 1 and 5 because its correctness function used asks the LLM to rate the correctness between 0 and 5 of the response. All other metrics return a value between 0 and 1 (no, yes), where 1 is the best score.

{
  "correctness": 5.0,
  "faithfulness": 1.0,
  "guideline": 1.0,
  "relevancy": 1.0
}


In [19]:
# Eval Templates can be found in the eval_parser.
templates = {
    "faithfulness": eval_parser.FAITHFULNESS_EVAL_TEMPLATE.format(),
    "relevancy": eval_parser.RELEVANCY_EVAL_TEMPLATE.format(),
    "correctness": eval_parser.CORRECTNESS_EVAL_TEMPLATE.format(),
    "guideline": eval_parser.GUIDELINE_EVAL_TEMPLATE.partial_format(guidelines=guidelines).format(),
}

print(json.dumps(templates, indent=2, sort_keys=True))

{
  "correctness": "\nYou are an expert evaluation system for a recipe recommender.\n\nYou are given the following information:\n- a user query, and\n- a generated answer, and\n- a context including valid recipe titles.\n\nYour job is to judge the relevance and correctness of the generated answer.\nOutput a single score that represents a holistic evaluation.\nYou must return your response in a line with only the score.\nDo not return answers in any other format.\nOn a separate line provide your reasoning for the score as well.\n\nRegarding the nutri score, the who score, and the fsa score: Those are between 0 and 1, where 0 is the worst and 1 is \nthe best. So, i.e., if the query asks for a recipe with a nutri score of 0.5, the answer should contain a recipe with a \nnutri score of 0.5 or higher.\n\nFollow these guidelines for scoring:\n- Your score has to be between 1 and 5, where 1 is the worst and 5 is the best.\n- If the generated answer is not relevant to the user query, you shoul